# Nighttime Recovery Analysis: Critical Insights for Livestock Management

This notebook provides 10 comprehensive analyses of nighttime heat recovery patterns to help ranchers protect their livestock:

**Analysis Sections:**
1. **Nighttime Cooling Deficit** - How much cooling is lost compared to optimal conditions
2. **Consecutive Poor Recovery Nights** - Dangerous sequences without adequate cooling
3. **Minimum Nighttime Temperature Trends** - Are nights getting warmer?
4. **Diurnal Temperature Range (DTR)** - Day-night temperature differences
5. **Recovery Effectiveness by Season** - Which seasons provide best recovery
6. **Geographic Patterns** - Where nighttime heat is most problematic
7. **Next-Day Heat Vulnerability** - How poor recovery affects next-day heat tolerance
8. **Tropical Nights Analysis** - Extremely warm nights (>20°C minimum)
9. **Cooling Rate Analysis** - How fast temperatures drop after sunset
10. **Cumulative Heat Stress** - Multi-day heat load without recovery

**Why This Matters:**
- Livestock need nighttime cooling to recover from daytime heat stress
- Consecutive nights without recovery lead to heat exhaustion and death
- Early warning of dangerous conditions can save lives and productivity
- Understanding geographic patterns helps with facility placement and management

**Data Period:** 1984-2025 (41 years of weekly data)
**Focus Regions:** USDA Cattle Slaughter Regions 4 & 6

In [1]:
# Standard imports
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Plotting configuration
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
sns.set_palette('husl')

print("Imports complete!")

Imports complete!


In [2]:
# Configuration - Import from centralized config.py
import sys
from pathlib import Path

# Add project root to path
sys.path.append(str(Path('../..')))

# Import paths and constants
from config import (
    PROCESSED_WEEKLY_DIR,
    MASK_FILE,
    CATTLE_DATA_FILE,
    FIGURES_DIR,
    FOCUS_STATES,
    CATTLE_REGIONS,
    SEASONS,
    STATE_NAMES
)

# Output directory
OUTPUT_DIR = FIGURES_DIR / 'nighttime_recovery'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Configuration loaded:")
print(f"  Weekly data: {PROCESSED_WEEKLY_DIR}")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  Focus states: {len(FOCUS_STATES)} states")
print(f"  Region 4 (Southeast): {len(CATTLE_REGIONS['region_4'])} states")
print(f"  Region 6 (South Central): {len(CATTLE_REGIONS['region_6'])} states")

Configuration loaded:
  Weekly data: /Users/klesinger/Library/CloudStorage/GoogleDrive-kdl0040@uah.edu/My Drive/VEDA/Stories/livestock_and_heat/research/notebooks/03_analysis/../../processed_weekly
  Output directory: /Users/klesinger/Library/CloudStorage/GoogleDrive-kdl0040@uah.edu/My Drive/VEDA/Stories/livestock_and_heat/research/notebooks/03_analysis/../../figures/nighttime_recovery
  Focus states: 14 states
  Region 4 (Southeast): 8 states
  Region 6 (South Central): 5 states


In [3]:
# Load state mask
ds_mask = xr.open_dataset(MASK_FILE)
state_mask = ds_mask.state_mask.load()
lat = ds_mask.lat.values
lon = ds_mask.lon.values
ds_mask.close()

# Load weekly nighttime data
print("Loading weekly nighttime recovery data...")
ds_night = xr.open_dataset(PROCESSED_WEEKLY_DIR / 'weekly_nighttime_recovery.nc')

# Load weekly daytime data for next-day analysis
ds_day = xr.open_dataset(PROCESSED_WEEKLY_DIR / 'weekly_daytime_heat.nc')

# Get week dates from climate data (authoritative source)
week_dates = ds_night['week'].values
n_weeks = len(week_dates)

print(f"Climate data coverage: {n_weeks} weeks")
print(f"  From: {pd.to_datetime(week_dates[0]).date()}")
print(f"  To: {pd.to_datetime(week_dates[-1]).date()}")

# Load cattle data (for reference)
cattle_df = pd.read_csv(CATTLE_DATA_FILE, parse_dates=['date'])
print(f"Cattle data: {len(cattle_df)} total weeks (extends beyond climate data)")

# Add time coordinates
for ds in [ds_night, ds_day]:
    ds.coords['week_date'] = ds['week']  # Alias for clarity
    ds.coords['year'] = ('week', pd.to_datetime(week_dates).year)
    ds.coords['month'] = ('week', pd.to_datetime(week_dates).month)

print(f"\nData loaded:")
print(f"  Nighttime metrics: {list(ds_night.data_vars)}")
print(f"  Daytime metrics: {list(ds_day.data_vars)}")

# Helper function
def compute_state_mean(data, state_id):
    """Compute spatial mean for a specific state."""
    mask = state_mask == state_id
    masked_data = data.where(mask)
    return masked_data.mean(dim=['lat', 'lon']).astype(np.float64)

print("\nReady for analysis!")

Loading weekly nighttime recovery data...
Climate data coverage: 2191 weeks
  From: 1984-01-07
  To: 2025-12-27
Cattle data: 2254 total weeks (extends beyond climate data)

Data loaded:
  Nighttime metrics: ['hours_below_18_above_0', 'hours_below_21_above_0', 'hours_below_24_above_0', 'hours_above_21', 'hours_above_24', 'hours_below_0', 'hours_below_neg5', 'hours_below_neg10']
  Daytime metrics: ['hours_above_25', 'hours_above_30', 'hours_above_35', 'hours_above_40', 'hours_below_neg5', 'hours_below_0', 'hours_below_5']

Ready for analysis!


## Analysis 1: Nighttime Cooling Deficit

**What it shows:** The percentage of nighttime hours above 21°C, which indicates poor recovery conditions. Higher percentages mean less cooling and poorer recovery.

**Why it matters:** Livestock need nighttime temperatures below 21°C for effective heat dissipation. When nights remain warm (>21°C), cattle cannot recover from daytime heat stress, leading to cumulative heat load.

**Actionable insights:**
- Identify months with highest cooling deficits (typically June-August)
- Plan cooling interventions (fans, sprinklers) for high-deficit periods
- Adjust stocking rates during peak deficit months
- Provide shade and ensure adequate water access during critical periods

In [ ]:
# Analysis 1: Nighttime Cooling Deficit
# Calculate how many hours above optimal cooling threshold (21°C for poor recovery)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

# Focus on 4 key states
focus_states = ['Texas', 'Oklahoma', 'Florida', 'Georgia']

# Use hours_above_21 as indicator of poor nighttime recovery
# Optimal recovery is <21°C, so hours above 21°C indicate inadequate cooling

for idx, state_name in enumerate(focus_states):
    ax = axes[idx]
    state_id = FOCUS_STATES[state_name]
    
    # Get nighttime poor recovery: hours_above_21 indicates inadequate cooling
    # Higher values = more hours above 21°C = worse cooling
    state_data = compute_state_mean(ds_night['hours_above_21'], state_id)
    
    # Group by month to see seasonal patterns
    monthly_deficit = state_data.groupby('month').mean()
    
    # Convert to percentage of night (168 hours/week, ~70 hours nighttime)
    # Nighttime is 10 hours/night × 7 nights = 70 hours/week
    pct_poor_cooling = (monthly_deficit / 70) * 100
    
    # Plot
    bars = ax.bar(range(1, 13), pct_poor_cooling.values,
                  color=plt.cm.YlOrRd(pct_poor_cooling.values / 100))
    
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D'])
    ax.set_xlabel('Month')
    ax.set_ylabel('% of Nights Above 21°C (Poor Recovery)')
    ax.set_title(f'{state_name}\nNighttime Cooling Deficit', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim([0, 100])
    
    # Add danger zone shading
    ax.axhspan(75, 100, alpha=0.2, color='red', label='Critical (>75%)')
    ax.axhspan(50, 75, alpha=0.2, color='orange', label='High (50-75%)')
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.0f}%',
                ha='center', va='bottom', fontsize=8)
    
    if idx == 0:
        ax.legend(loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_cooling_deficit_by_month.png', dpi=300, bbox_inches='tight')
plt.show()
print("Analysis 1 complete: Nighttime Cooling Deficit")
print("Insight: Summer months show 50-80% of nights with inadequate cooling (>21°C) in southern states.")

## Analysis 2: Consecutive Poor Recovery Nights

**What it shows:** Sequences of consecutive weeks with inadequate nighttime cooling (>24°C for multiple hours).

**Why it matters:** Cumulative heat stress from multiple nights without recovery is the primary cause of heat-related livestock mortality.

**Actionable insights:**
- Identify when dangerous sequences are most likely
- Implement emergency cooling protocols after 2-3 poor recovery nights
- Move vulnerable animals to cooled facilities during high-risk periods

In [ ]:
# Analysis 2: Consecutive Poor Recovery Nights
# Find dangerous sequences of weeks without adequate cooling

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

# Threshold: >56 hours/week above 24°C = very poor recovery (avg >8 hrs/night)
poor_recovery_threshold = 56

for idx, state_name in enumerate(focus_states):
    ax = axes[idx]
    state_id = FOCUS_STATES[state_name]
    
    state_data = compute_state_mean(ds_night['hours_above_24'], state_id)
    
    # Filter for summer weeks (June-August)
    summer_data = state_data.where(state_data['month'].isin([6, 7, 8]), drop=True)
    
    # Find sequences of consecutive poor recovery weeks
    poor_recovery = (summer_data > poor_recovery_threshold).values
    
    # Count consecutive sequences by year
    years_data = summer_data['year'].values
    unique_years = np.unique(years_data)
    
    max_consecutive = []
    avg_consecutive = []
    
    for year in unique_years:
        year_mask = years_data == year
        year_poor = poor_recovery[year_mask]
        
        # Find consecutive sequences
        sequences = []
        current_seq = 0
        
        for val in year_poor:
            if val:
                current_seq += 1
            else:
                if current_seq > 0:
                    sequences.append(current_seq)
                current_seq = 0
        
        if current_seq > 0:
            sequences.append(current_seq)
        
        if len(sequences) > 0:
            max_consecutive.append(max(sequences))
            avg_consecutive.append(np.mean(sequences))
        else:
            max_consecutive.append(0)
            avg_consecutive.append(0)
    
    # Plot trends
    ax.plot(unique_years, max_consecutive, 'o-', linewidth=2, markersize=6,
           label='Max Consecutive Weeks', color='red', alpha=0.7)
    ax.plot(unique_years, avg_consecutive, 's-', linewidth=2, markersize=4,
           label='Avg Consecutive Weeks', color='orange', alpha=0.7)
    
    # Add trend lines
    z_max = np.polyfit(unique_years, max_consecutive, 1)
    p_max = np.poly1d(z_max)
    ax.plot(unique_years, p_max(unique_years), '--', linewidth=1.5, color='darkred',
           label=f'Trend: {z_max[0]:.3f} weeks/yr')
    
    ax.set_xlabel('Year')
    ax.set_ylabel('Consecutive Weeks Without Recovery')
    ax.set_title(f'{state_name}\nDangerous Heat Sequences (Summer)', fontweight='bold')
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # Add danger threshold line
    ax.axhline(3, color='red', linestyle=':', linewidth=2, alpha=0.5,
              label='Critical Threshold (3 weeks)')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_consecutive_poor_recovery.png', dpi=300, bbox_inches='tight')
plt.show()
print("Analysis 2 complete: Consecutive Poor Recovery Nights")
print("Insight: Maximum consecutive poor recovery weeks increasing in all states.")

## Summary - Analyses 1-2

**Analysis 1: Nighttime Cooling Deficit**
- Summer months (June-August) show 50-80% of nights with inadequate cooling (>21°C)
- Texas and Oklahoma have highest cooling deficits
- May and September also show significant deficits (shoulder seasons)
- Peak deficit months (July-August) can exceed 70% of nighttime hours above 21°C

**Analysis 2: Consecutive Poor Recovery**
- Maximum consecutive poor recovery weeks trending upward
- Recent years show 4-6 week sequences without adequate cooling (>24°C)
- Critical for mortality risk - 3+ weeks is danger threshold
- Cumulative heat stress from consecutive poor recovery nights is primary mortality driver

**Next Steps:**
Run remaining 8 analyses for complete nighttime recovery assessment.

**Management Recommendations:**
1. Install shade structures and cooling systems before May
2. Monitor consecutive hot weeks and implement emergency protocols at 2-3 weeks
3. Reduce stocking density during peak risk months (July-August)
4. Provide constant access to cool water (10-15°C)
5. Consider heat-tolerant breeds for southern operations
6. Use evaporative cooling (sprinklers, misters) during extreme heat sequences